# 07 — פונקציית ההתאמה: קוסינוס מוסווה עם כיווץ

**המטרה:** בהינתן תשובות של משתמש, לדרג את 14 המפלגות. הקלט הוא `party_positions.json` ממחברת 06.

**ההחלטות:**
- ההשוואה נעשית ברמת **השאלה** (25 ממדים), לא ברמת הציר. המשתמש והמפלגה נמדדים ביחס לאותו משפט, ולכן לא צריך `polarity`.
- **קוסינוס מוסווה:** מחשבים רק על שאלות שהמשתמש ענה עליהן **וגם** המפלגה מכוסה בהן.
- **כיווץ:** `score = cos × n / (n + k)`, כאשר `n` הוא מספר השאלות המשותפות.
- **משקל לשאלה:** משקל הבסיס של השאלה (`weight` ב-quiz.json), כפול 2 אם המשתמש סימן אותה כחשובה. אותו כלל כמו באתר היום (`IMPORTANT_MULTIPLIER` ב-`web/lib/cosine.ts`).

**הפורמט של תשובות משתמש בכל המחברת:** מילון מ-`qid` לזוג `(value, important)`, כאשר `value` הוא מ-1 עד 5 ו-3 הוא ניטרלי. שאלה שהמשתמש דילג עליה לא מופיעה במילון:

```python
{"q01": (5, False), "q07": (2, True)}
```

כל משימה היא פונקציה אחת, ואחריה תא בדיקה עם מקרים קטנים שאפשר לחשב בראש.

In [2]:
import json, math, random
import pandas as pd

positions = json.load(open("party_positions.json", encoding="utf-8"))
questions = pd.DataFrame(json.load(open("../data/seed/quiz.json", encoding="utf-8"))["questions"]).set_index("id")
BASE_WEIGHT = questions["weight"].to_dict()
IMPORTANT_MULTIPLIER = 2
len(positions), len(BASE_WEIGHT)

(14, 25)

## משימה 1 — מתשובות לווקטור ולמשקלים

שתי פונקציות קטנות:

- `answers_to_vector(answers)` מחזירה `{qid: position}`, כאשר `position = (value − 3) / 2`. כך 1 הופך ל-‎−1, ‏3 ל-0 ו-5 ל-1.
- `answers_to_weights(answers)` מחזירה `{qid: weight}`, כלומר משקל הבסיס כפול `IMPORTANT_MULTIPLIER` אם `important`.

**רמז:** dict comprehension. כשעוברים על `answers.items()`, כל איבר הוא `qid, (value, important)`. אפשר לפרק את הזוג ישר בתוך ה-`for`.

In [3]:
# TODO
def answers_to_vector(answers):
    res = {}
    for key, (value, important) in answers.items():
        res[key] = (value-3) / 2
    return res

def answers_to_weights(answers):
    res = {}
    for key, (value, important) in answers.items():
        res[key] = BASE_WEIGHT[key] * (IMPORTANT_MULTIPLIER if important else 1)
    return res

In [4]:
# בדיקה
assert answers_to_vector({"q01": (1, False), "q02": (3, False), "q03": (4, True)}) == {"q01": -1.0, "q02": 0.0, "q03": 0.5}
w = answers_to_weights({"q01": (5, False), "q03": (5, True)})
assert w["q01"] == BASE_WEIGHT["q01"] and w["q03"] == BASE_WEIGHT["q03"] * 2, w
print("עבר")

עבר


## משימה 2 — קוסינוס מוסווה וממושקל

`masked_cosine(user, party, weights)` מחזירה **זוג**: `(cos, n)`.

1. `shared`: השאלות שמופיעות גם ב-`user` וגם ב-`party`. רמז: חיתוך של `set`.
2. על השאלות ב-`shared` בלבד:
   - `dot  = Σ w·u·p`
   - `norm_u = √(Σ w·u²)`
   - `norm_p = √(Σ w·p²)`
3. `cos = dot / (norm_u × norm_p)`. **אבל** אם אחד הנורמות הוא 0, מחזירים `cos = 0`. מתי זה קורה? למשל, כשאין אף שאלה משותפת, או כשהמשתמש ענה 3 על כל השאלות המשותפות.
4. `n = len(shared)`

**למה המשקל נכנס גם לנורמות ולא רק ל-`dot`?** אחרת שאלה חשובה הייתה מגדילה רק את המונה, והקוסינוס היה יכול לצאת יותר מ-1.

In [7]:
# TODO
def masked_cosine(user, party, weights):
    shared = set(user) & set(party)
    dot, sum_u, sum_p = 0, 0, 0
    for q in shared:
        w, u, p = weights[q], user[q], party[q]
        dot += w*u*p
        sum_u += w * u ** 2
        sum_p += w * p ** 2
    norm_u, norm_p = math.sqrt(sum_u), math.sqrt(sum_p)
    if norm_p == 0 or norm_u == 0   :
        return 0.0, len(shared)
    return dot / (norm_u * norm_p), len(shared)

In [8]:
# בדיקה — המקרים שחישבנו בשיחה
one = {"q01": 1.0}
assert masked_cosine({"q01": 0.5}, {"q01": 1.0}, one) == (1.0, 1)
assert masked_cosine({"q01": -0.5}, {"q01": 1.0}, one) == (-1.0, 1)
# שאלה שרק למשתמש יש לא משתתפת
assert masked_cosine({"q01": 1.0, "q02": -1.0}, {"q01": 1.0}, {"q01": 1, "q02": 1}) == (1.0, 1)
# אין שאלה משותפת → 0, ולא שגיאה של חלוקה באפס
assert masked_cosine({"q01": 1.0}, {"q02": 1.0}, one) == (0.0, 0)
# ניטרלי בכל המשותפות → 0
assert masked_cosine({"q01": 0.0}, {"q01": 1.0}, one) == (0.0, 1)
# משקל: שאלה כבדה שמסכימים עליה מושכת למעלה
c_light, _ = masked_cosine({"q01": 1, "q02": 1}, {"q01": 1, "q02": -1}, {"q01": 1, "q02": 1})
c_heavy, _ = masked_cosine({"q01": 1, "q02": 1}, {"q01": 1, "q02": -1}, {"q01": 3, "q02": 1})
assert abs(c_light) < 1e-9 and c_heavy > 0.4, (c_light, c_heavy)
print("עבר")

עבר


## משימה 3 — כיווץ, ומציון לאחוזים

- `shrink(cos, n, k)` מחזירה `cos × n / (n + k)`. מה קורה כש-`n = 0` ו-`k = 0`? ודא שהפונקציה מחזירה 0 ולא נופלת.
- `to_percent(score)` ממירה מ-[‎−1, 1] ל-[0, 100], בדיוק כמו באתר היום. כאן 0, כלומר "לא יודעים", הופך ל-50%.

In [9]:
# TODO
def shrink(cos, n, k):
    return cos * n / (n + k) if n + k != 0 else 0

def to_percent(score):
    return (score + 1) * 50

In [10]:
# בדיקה
assert shrink(1.0, 1, 3) == 0.25
assert shrink(0.75, 15, 3) == 0.75 * 15 / 18
assert shrink(1.0, 5, 0) == 1.0
assert shrink(0.0, 0, 0) == 0.0
assert to_percent(-1) == 0 and to_percent(0) == 50 and to_percent(1) == 100
print("עבר")

עבר


## משימה 4 — לדרג את כל המפלגות

`match(answers, k)` מחזירה DataFrame עם שורה לכל מפלגה, האינדקס הוא שם המפלגה, והעמודות הן `cos`, ‏`n`, ‏`score` ו-`pct`. הטבלה ממוינת לפי `score` מהגבוה לנמוך.

זו פשוט הרכבה של שלוש המשימות הקודמות: ממירים את התשובות פעם אחת, ואז עוברים בלולאה על `positions.items()`.

**רמז:** רשימה של מילונים, שורה לכל מפלגה, ואז `pd.DataFrame(rows).set_index("party")`.

In [13]:
# TODO
def match(answers, k=3):
    user = answers_to_vector(answers)
    weights = answers_to_weights(answers)
    rows = []
    for party, p in positions.items():
        cos, n = masked_cosine(user,p,weights)
        score = shrink(cos,n,k)
        rows.append({"party": party, "cos": cos, "n": n, "score": score, "pct": to_percent(score)})
    return pd.DataFrame(rows).set_index("party").sort_values("score", ascending=False)    

In [14]:
# בדיקה
r = match({"q01": (5, False), "q02": (5, False)}, k=3)
assert list(r.columns) == ["cos", "n", "score", "pct"], list(r.columns)
assert len(r) == 14 and r["score"].is_monotonic_decreasing
assert r.loc["the-democrats", "n"] == 1 and r.loc["yisrael-beitenu", "n"] == 0
r

,cos,n,score,pct
party,,,,
the-democrats,1.000000,1,0.250000,62.500000
yashar,1.000000,1,0.250000,62.500000
likud,0.236525,2,0.094610,54.730499
religious-zionism,0.236525,2,0.094610,54.730499
shas,0.236525,2,0.094610,54.730499
yahadut-hatorah,0.236525,2,0.094610,54.730499
beyachad,0.000000,0,0.000000,50.000000
joint-list,0.000000,0,0.000000,50.000000
kachol-lavan,0.000000,0,0.000000,50.000000


## משימה 5 — האם כל מפלגה מוצאת את עצמה?

ניסוי: לכל מפלגה, בונים משתמש **שעונה בדיוק כמו המפלגה**, וזה הכול. משתמש כזה עונה רק על השאלות שהמפלגה מכוסה בהן, כאשר `value = 3 + 2 × position`, כך ש-1 הופך ל-5, ‏0.5 ל-4 ו-0 ל-3. אחר כך בודקים אם המפלגה יוצאת ראשונה ב-`match`.

כתוב `self_recovery(k)` שמחזירה טבלה: לכל מפלגה, מי יצאה ראשונה, האם זו המפלגה עצמה, ומה הציונים של שתיהן.

**הרץ אותה עם `k=0` ועם `k=3`, ולפני שאתה מסתכל על התוצאות, תנחש:** כמה מפלגות ימצאו את עצמן בכל אחד מהמקרים? ואם יש כישלונות, למה?

In [17]:
# TODO
def self_recovery(k):
    rows = []
    for party, p in positions.items():
        answers = {q: (3 + 2 * v, False) for q, v in p.items()}
        r = match(answers, k)
        rows.append({
            "party": party,
            "winner": r.index[0],                # מי יצאה ראשונה
            "found": r.index[0] == party,                        # האם זו המפלגה עצמה? (True/False)
            "own": r.loc[party, "score"],        # הציון של המפלגה עצמה
            "best": r.iloc[0]["score"],          # הציון של המנצחת
        })
    return pd.DataFrame(rows).set_index("party")

In [21]:
# בדיקה
r3 = self_recovery(0)
assert len(r3) == 14
r3

,winner,found,own,best
party,,,,
amcha-israel,amcha-israel,True,1.0,1.0
beyachad,beyachad,True,1.0,1.0
joint-list,joint-list,True,1.0,1.0
kachol-lavan,religious-zionism,False,1.0,1.0
likud,likud,True,1.0,1.0
otzma-yehudit,otzma-yehudit,True,1.0,1.0
raam,kachol-lavan,False,1.0,1.0
religious-zionism,kachol-lavan,False,1.0,1.0
shas,shas,True,1.0,1.0


In [ ]:
self_recovery(0)

## שאלות לחשיבה (בלי קוד)

1. ב-`k=0` חלק מהמפלגות "נגנבו" על ידי מפלגה אחרת. תסתכל על הציון של המנצחת ועל ה-`n` שלה. מה קרה, במונחים של מה שדיברנו עליו (ממד אחד)?
2. ‏`k=3` פתר את הניסוי הזה. האם זה אומר ש-3 הוא הערך הנכון? מה הניסוי הזה **לא** בודק? רמז: המשתמשים כאן "מושלמים". רוב המשתמשים האמיתיים לא נראים כמו מפלגה.
3. הרץ `match` על משתמש אקראי, למשל `{q: (random.randint(1, 5), False) for q in BASE_WEIGHT}`, כמה פעמים. מי מופיעה הרבה למעלה? יש לה הרבה עמדות או מעט?

שאלה 3 היא שלב ג: אבחון ההטיה.